# IMS OL-vs-DA Tables and Maps (from precomputed script outputs)

This notebook is lightweight by design: it reads outputs written by `run_ims_ol_da_cell_metrics.py` and focuses on quick analysis and plotting.

Outputs expected:
- `ims_ol_da_comparison_table_*.parquet/csv`
- `ims_ol_da_pair_daily_*.parquet/csv`
- `ims_ol_da_scope_metadata_*.csv`
- `ims_ol_da_cell_counts_metrics_*.nc4`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
from cartopy.feature import ShapelyFeature

SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

DOMAIN = "SMAP_EASEv2_M36_GLOBAL"
YEAR_START = 2000
YEAR_END = 2024
SCF_THRESHOLD = 0.5
MIN_IMS_SNOW_DAYS = 10

OUTPUT_DIR = Path("/discover/nobackup/projects/land_da/geosldas-analysis/projects/IMS/output")
CACHE_TAG = (
    f"{DOMAIN}_{YEAR_START}_{YEAR_END}_thr{SCF_THRESHOLD:.2f}"
    f"_imsSnowDaysGe{MIN_IMS_SNOW_DAYS}"
).replace(".", "p")

COMPARISON_PARQUET = OUTPUT_DIR / f"ims_ol_da_comparison_table_{CACHE_TAG}.parquet"
COMPARISON_CSV = OUTPUT_DIR / f"ims_ol_da_comparison_table_{CACHE_TAG}.csv"
PAIR_DAILY_PARQUET = OUTPUT_DIR / f"ims_ol_da_pair_daily_{CACHE_TAG}.parquet"
PAIR_DAILY_CSV = OUTPUT_DIR / f"ims_ol_da_pair_daily_{CACHE_TAG}.csv"
SCOPE_META_CSV = OUTPUT_DIR / f"ims_ol_da_scope_metadata_{CACHE_TAG}.csv"
CELL_COUNTS_NC = OUTPUT_DIR / f"ims_ol_da_cell_counts_metrics_{CACHE_TAG}.nc4"

# Use local Natural Earth cache when available to avoid downloader issues.
CARTOPY_DATA_DIR_CANDIDATES = [
    Path("/home/amfox/.local/share/cartopy"),
    Path.home() / ".local" / "share" / "cartopy",
]
CARTOPY_DATA_DIR = next((d for d in CARTOPY_DATA_DIR_CANDIDATES if d.exists()), CARTOPY_DATA_DIR_CANDIDATES[0])
cartopy.config["data_dir"] = str(CARTOPY_DATA_DIR)
NATURAL_EARTH_ROOT = CARTOPY_DATA_DIR / "shapefiles" / "natural_earth"

print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"COMPARISON={COMPARISON_PARQUET if COMPARISON_PARQUET.exists() else COMPARISON_CSV}")
print(f"PAIR_DAILY={PAIR_DAILY_PARQUET if PAIR_DAILY_PARQUET.exists() else PAIR_DAILY_CSV}")
print(f"SCOPE_META={SCOPE_META_CSV}")
print(f"CELL_COUNTS_NC={CELL_COUNTS_NC}")
print(f"CARTOPY_DATA_DIR={CARTOPY_DATA_DIR}")


In [ ]:
if COMPARISON_PARQUET.exists():
    comparison_df = pd.read_parquet(COMPARISON_PARQUET)
else:
    comparison_df = pd.read_csv(COMPARISON_CSV)

if PAIR_DAILY_PARQUET.exists():
    pair_daily = pd.read_parquet(PAIR_DAILY_PARQUET)
else:
    pair_daily = pd.read_csv(PAIR_DAILY_CSV)

scope_meta = pd.read_csv(SCOPE_META_CSV)

print("comparison_df:", comparison_df.shape)
print("pair_daily:", pair_daily.shape)
print("scope_meta:", scope_meta.shape)

all_period_tbl = comparison_df[comparison_df["scope"] == "ALL_PERIOD"].copy()
all_period_tbl = all_period_tbl.set_index("metric").reindex([
    "accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate"
]).reset_index()
all_period_tbl


In [ ]:
def fmt_ci(v, lo, hi):
    if np.isfinite(v) and np.isfinite(lo) and np.isfinite(hi):
        return f"{v:.3f} [{lo:.3f}, {hi:.3f}]"
    if np.isfinite(v):
        return f"{v:.3f}"
    return "NaN"

all_period_ci = comparison_df[comparison_df["scope"] == "ALL_PERIOD"].copy()
all_period_ci = all_period_ci.set_index("metric").reindex(SEASON_ORDER if False else [
    "accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate"
]).reset_index()

all_period_ci["OL (95% CI)"] = [
    fmt_ci(v, lo, hi)
    for v, lo, hi in zip(all_period_ci["ol"], all_period_ci["ol_ci_lo"], all_period_ci["ol_ci_hi"])
]
all_period_ci["DA (95% CI)"] = [
    fmt_ci(v, lo, hi)
    for v, lo, hi in zip(all_period_ci["da"], all_period_ci["da_ci_lo"], all_period_ci["da_ci_hi"])
]

all_period_ci_simple = all_period_ci[["metric", "OL (95% CI)", "DA (95% CI)"]].copy()
all_period_ci_simple


In [ ]:
ds = xr.open_dataset(CELL_COUNTS_NC)

print(ds)

ny = int(ds.attrs["grid_ny"])
nx = int(ds.attrs["grid_nx"])
cell_i = ds["cell_i"].values.astype(np.int64)
cell_j = ds["cell_j"].values.astype(np.int64)

EXP_INDEX = {"OL": 0, "DA": 1}
METRICS = ["accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate"]

scope_df = scope_meta.copy()
scope_df


In [ ]:
def _load_local_feature(shp_path: Path | None, *, facecolor="none", edgecolor="black", linewidth=0.5):
    if shp_path is None:
        return None
    shp_path = Path(shp_path)
    if not shp_path.exists():
        return None
    geoms = list(shapereader.Reader(str(shp_path)).geometries())
    if not geoms:
        return None
    return ShapelyFeature(
        geoms,
        ccrs.PlateCarree(),
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=linewidth,
    )


def build_local_features():
    coast = NATURAL_EARTH_ROOT / "physical" / "ne_50m_coastline.shp"
    borders = NATURAL_EARTH_ROOT / "cultural" / "ne_50m_admin_0_boundary_lines_land.shp"
    land = NATURAL_EARTH_ROOT / "physical" / "ne_50m_land.shp"
    return {
        "land": _load_local_feature(land, facecolor="0.95", edgecolor="none", linewidth=0.0),
        "coast": _load_local_feature(coast, facecolor="none", edgecolor="0.2", linewidth=0.5),
        "borders": _load_local_feature(borders, facecolor="none", edgecolor="0.3", linewidth=0.4),
    }


local_features = build_local_features()


def add_local_map_features(ax):
    if local_features.get("land") is not None:
        ax.add_feature(local_features["land"], zorder=0)
    if local_features.get("coast") is not None:
        ax.add_feature(local_features["coast"], zorder=3)
    if local_features.get("borders") is not None:
        ax.add_feature(local_features["borders"], zorder=3)


def find_scope_id(scope: str, year=None, season=None) -> int:
    sub = scope_df[scope_df["scope"] == scope].copy()
    if year is None:
        sub = sub[sub["year"] == -1]
    else:
        sub = sub[sub["year"] == int(year)]
    if season is None:
        sub = sub[sub["season"] == "ALL"]
    else:
        sub = sub[sub["season"] == str(season)]
    if sub.empty:
        raise KeyError(f"No scope row found for scope={scope}, year={year}, season={season}")
    return int(sub.iloc[0]["scope_id"])


def values_to_grid(values_1d: np.ndarray) -> np.ndarray:
    g = np.full((ny, nx), np.nan, dtype=np.float32)
    g[cell_j, cell_i] = np.asarray(values_1d, dtype=np.float32)
    return g


def _fill_nonfinite_2d(arr2d: np.ndarray) -> np.ndarray:
    out = np.asarray(arr2d, dtype=np.float64).copy()
    ny0, nx0 = out.shape

    x = np.arange(nx0, dtype=np.float64)
    for j in range(ny0):
        row = out[j, :]
        good = np.isfinite(row)
        if np.sum(good) < 2:
            continue
        if np.sum(good) < nx0:
            out[j, :] = np.interp(x, x[good], row[good])

    y = np.arange(ny0, dtype=np.float64)
    for i in range(nx0):
        col = out[:, i]
        good = np.isfinite(col)
        if np.sum(good) < 2:
            continue
        if np.sum(good) < ny0:
            out[:, i] = np.interp(y, y[good], col[good])

    if np.any(~np.isfinite(out)):
        g = np.nanmean(out)
        out[~np.isfinite(out)] = 0.0 if not np.isfinite(g) else g

    return out.astype(np.float32)


def make_finite_lonlat_for_pcolormesh(lon2d: np.ndarray, lat2d: np.ndarray):
    lon = ((np.asarray(lon2d, dtype=np.float32) + 180.0) % 360.0) - 180.0
    lat = np.asarray(lat2d, dtype=np.float32)
    if np.all(np.isfinite(lon)) and np.all(np.isfinite(lat)):
        return lon, lat
    return _fill_nonfinite_2d(lon), _fill_nonfinite_2d(lat)


def get_metric_grid(metric: str, experiment: str, scope: str, year=None, season=None) -> np.ndarray:
    exp_i = EXP_INDEX[experiment]
    sid = find_scope_id(scope, year=year, season=season)
    vals = ds[metric].isel(experiment=exp_i, scope=sid).values
    return values_to_grid(vals)


def get_delta_grid(metric: str, scope: str, year=None, season=None) -> np.ndarray:
    g_da = get_metric_grid(metric, "DA", scope, year=year, season=season)
    g_ol = get_metric_grid(metric, "OL", scope, year=year, season=season)
    return g_da - g_ol


cell_lon_grid = values_to_grid(ds["cell_lon"].values)
cell_lat_grid = values_to_grid(ds["cell_lat"].values)
lon_plot, lat_plot = make_finite_lonlat_for_pcolormesh(cell_lon_grid, cell_lat_grid)


def _plot_one_panel(ax, data2d, title, cmap, vmin=None, vmax=None, norm=None):
    m = ax.pcolormesh(
        lon_plot,
        lat_plot,
        data2d,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        norm=norm,
        shading="auto",
        rasterized=True,
    )
    ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())
    add_local_map_features(ax)
    ax.gridlines(draw_labels=False, linewidth=0.25, color="0.6", alpha=0.4, linestyle="--")
    ax.set_title(title, fontsize=11)
    return m


def plot_all_period_metric_3x1(metric: str):
    proj = ccrs.Robinson()
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), subplot_kw={"projection": proj}, constrained_layout=True)

    g_ol = get_metric_grid(metric, "OL", "ALL_PERIOD")
    g_da = get_metric_grid(metric, "DA", "ALL_PERIOD")
    g_d = g_da - g_ol

    m0 = _plot_one_panel(axes[0], g_ol, f"{metric}: OL (ALL_PERIOD)", cmap="viridis", vmin=0.0, vmax=1.0)
    m1 = _plot_one_panel(axes[1], g_da, f"{metric}: DA (ALL_PERIOD)", cmap="viridis", vmin=0.0, vmax=1.0)
    nd = TwoSlopeNorm(vmin=-0.5, vcenter=0.0, vmax=0.5)
    m2 = _plot_one_panel(axes[2], g_d, f"{metric}: DA - OL (ALL_PERIOD)", cmap="RdBu_r", norm=nd)

    cb0 = fig.colorbar(m0, ax=axes[0], orientation="horizontal", pad=0.03, fraction=0.05)
    cb0.set_label("fraction")
    cb1 = fig.colorbar(m1, ax=axes[1], orientation="horizontal", pad=0.03, fraction=0.05)
    cb1.set_label("fraction")
    cb2 = fig.colorbar(m2, ax=axes[2], orientation="horizontal", pad=0.03, fraction=0.05)
    cb2.set_label("DA - OL")

    fig.suptitle(f"IMS comparison: {metric} (All period)", fontsize=14)
    plt.show()


def plot_metric_delta_season_2x2(metric: str):
    proj = ccrs.Robinson()
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), subplot_kw={"projection": proj}, constrained_layout=True)
    nd = TwoSlopeNorm(vmin=-0.5, vcenter=0.0, vmax=0.5)
    for ax, season in zip(axes.ravel(), SEASON_ORDER):
        g = get_delta_grid(metric, "SEASON_ALL_YEARS", season=season)
        m = _plot_one_panel(ax, g, f"{season}", cmap="RdBu_r", norm=nd)
        fig.colorbar(m, ax=ax, orientation="horizontal", pad=0.03, fraction=0.05).set_label("DA - OL")

    fig.suptitle(f"{metric}: Seasonal DA - OL", fontsize=14)
    plt.show()


In [ ]:
# All-period map: accuracy for OL, DA, and DA-OL in a 3x1 Robinson figure.
plot_all_period_metric_3x1("accuracy")


In [ ]:
# Seasonal DA-OL maps (2x2 by season) for each metric.
for metric in METRICS:
    plot_metric_delta_season_2x2(metric)

# Compact all-period OL/DA table with CI strings.
all_period_ci_simple
